# CachingStrategies

**Module 12 · Lesson 12.4**  
*Netsetos GenAI Engineering Course v2.0*

**In this lesson you will:**
- The Economics: Why Cache At All
- Exact-Match Cache: Hash the Normalized Prompt
- Semantic Cache: Catch Paraphrases, Mind the False Positives
- Provider Prompt Caching: Cache the Prefix, 90 Percent Off Input
- TTL and Eviction: Bound the Cache
- Invalidation and Staleness: The Hard Problem

---

> This notebook is generated from the published lesson HTML. The HTML is the source of truth - do not hand-edit this notebook. Re-run the generator if the lesson updates.


In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install "anthropic>=0.40.0" python-dotenv -q  # noqa


In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("ANTHROPIC_API_KEY", "")    # console.anthropic.com


## The same questions, paid for again and again

> 💡 **Info**
>
> Open your production logs for a single day and a pattern jumps out: the same handful of questions, over and over - “what is the refund policy,” “how do I reset my password,” “what are your hours.” And every single time, your app pays **full price to an LLM** - a slow, dollar-costing round trip - to generate an answer it already generated an hour ago. **Caching is the fix**, and it is the single highest-leverage cost and latency win you have: turn a repeated call into a near-free lookup. But caching an LLM is trickier than caching a web page, because the same question arrives worded a hundred ways, because a cached answer can quietly go **stale and still look right**, and because some answers must **never be cached at all**. This lesson builds the full toolkit, and you can run every piece with no API key.

### 🎯 What you will be able to do after this lesson

- **Build** the caching layers - an exact-match cache, a semantic cache with a similarity threshold, an LRU+TTL cache, and a composed multi-tier stack - as runnable code.

- **Compare** exact vs semantic vs provider prompt caching, and reason about the hit-rate targets and the economics.

- **Operate** a semantic cache’s threshold and false-positive rate, and a cache’s scope + version keys and invalidation.

- **Evaluate** when to cache and when NOT to (personalized, time-sensitive, stateful, high-stakes).

> 📦 **Info**
>
> ✅ Before you startThis builds the layer **12.3** named as a first-class gateway feature and a rung of the degradation ladder, and it reuses **11.4**’s embeddings and cosine similarity to catch paraphrases. Scaling the cache backend (Redis cluster, Memorystore) is **Lesson 12.7**, and monitoring the hit rate and cost is **Lesson 12.8**. The embedding model that most determines semantic-cache quality is a Module 8 topic.

## 60-Second Warm-Up

Flip each card and answer before revealing.

> ☕ **Analogy**
>
> A cache is a **barista who remembers your usual order**. The first time, you explain exactly what you want and wait while they make it. But a regular does not re-explain every morning - the barista sees you, remembers, and the coffee is ready in seconds. A cache is that memory for your LLM: the first time a question is asked, you pay the full cost of generating an answer; after that, you serve the remembered one instantly and nearly free. **Where it breaks down:** a barista knows YOUR usual and would never hand you someone else’s drink - but a loose semantic cache might, and the menu (your data) changes, so the remembered answer can go stale. That is the whole art of this lesson.

> 📦 **Info**
>
> 🚫 Two misconceptions this lesson kills
> - **“Cache everything to save money.”** Personalized, time-sensitive, stateful, and high-stakes answers must NOT be cached (or need heavy invalidation) - and a hit rate above roughly 60 percent means you may not need an LLM at all.
> - **“Exact-match caching is enough.”** It misses every paraphrase, and most repeated questions are worded differently - which is exactly why semantic caching exists.

> 💡 **Info**
>
> ⚠️ Anti-patternSetting a **loose similarity threshold** to squeeze out more hits, and using an **un-versioned cache key**. A loose threshold returns a cached answer to a genuinely different question - a wrong answer that looks right. And a key that ignores the model, prompt, and source version leaves stale entries that keep serving a plausible-but-outdated answer after you upgrade the model or edit the source. Tune to a low false-positive rate, and version every key.

---

## 🎯 Concept 1: The Economics: Why Cache At All

### The Economics: Why Cache At All

A cache turns a repeated LLM call into a near-free lookup - the highest-leverage cost and latency win. But there are hit-rate targets that tell you when it pays off, and when it does not.

Start with the *why*, because it sets every later decision. An LLM call is slow (hundreds of milliseconds to seconds) and costs real money per token; a cache lookup is near-instant and near-free. So caching the **fraction of requests that repeat** is pure profit: each cache hit is an LLM call you did not pay for and a user who did not wait. The lever is the **hit rate** - the share of requests served from cache - and the practitioner targets are worth memorizing. Aim for a **25 to 45 percent hit rate**: real savings without over-caching. A hit rate **above roughly 60 percent** is a red flag, not a trophy - the workload is so repetitive you may not need an LLM at all. And for a very low-traffic endpoint (under about a thousand requests a day), the caching machinery can cost more to build and run than it saves. The block models the economics, keyless.

> 💰 **Analogy**
>
> It is a **photocopier next to a scribe**. Hiring a scribe to hand-write each copy of a document is slow and expensive - that is the LLM. A photocopier makes the second copy in a second for a fraction of a cent - that is the cache. The more copies of the *same* document you need (a high hit rate), the more the photocopier saves. But if every request is a genuinely new document, the copier sits idle and you are just paying for the machine.

Your cache reaches a 70 percent hit rate. Is that a good thing to celebrate?

**📝 `01_economics.py`** - *runnable*

In [ ]:
# THE ECONOMICS - a cache turns a repeated LLM call into a near-free lookup. It is the single
# highest-leverage cost and latency win, WITHIN limits: aim for a 25-45% hit rate. Prices and
# latencies are illustrative; the arithmetic is real. No key.

REQUESTS = 1000
LLM_COST, CACHE_COST = 0.0020, 0.00001       # $ per call (illustrative)
LLM_MS,   CACHE_MS   = 800, 5                 # latency per call (illustrative)

def model(hit_rate):
    hits = round(REQUESTS * hit_rate)
    misses = REQUESTS - hits                   # only misses actually call the LLM
    cost = misses * LLM_COST + hits * CACHE_COST
    base_cost = REQUESTS * LLM_COST            # cost with NO cache
    avg_ms = (misses * LLM_MS + hits * CACHE_MS) / REQUESTS
    base_ms = LLM_MS
    return hits, cost, base_cost, avg_ms, base_ms

print("1,000 requests, cost/latency vs cache hit rate:")
for h in (0.0, 0.30, 0.45, 0.67):
    hits, cost, base, avg_ms, base_ms = model(h)
    print(f"  hit rate {h:>4.0%}: {hits:>3} served from cache -> ${cost:6.3f} "
          f"(vs ${base:.2f}, {1-cost/base:.0%} saved), avg {avg_ms:.0f}ms (vs {base_ms}ms)")

print("\nRules of thumb:")
print("  target 25-45% hit rate: real savings without over-caching")
print("  above ~60%: the workload is so repetitive you may not need an LLM at all")
print("  under ~1,000 requests/day: the caching engineering can cost more than it saves")
# Output:
# 1,000 requests, cost/latency vs cache hit rate:
#   hit rate   0%:   0 served from cache -> $ 2.000 (vs $2.00, 0% saved), avg 800ms (vs 800ms)
#   hit rate  30%: 300 served from cache -> $ 1.403 (vs $2.00, 30% saved), avg 562ms (vs 800ms)
#   hit rate  45%: 450 served from cache -> $ 1.105 (vs $2.00, 45% saved), avg 442ms (vs 800ms)
#   hit rate  67%: 670 served from cache -> $ 0.667 (vs $2.00, 67% saved), avg 267ms (vs 800ms)
#
# Rules of thumb:
#   target 25-45% hit rate: real savings without over-caching
#   above ~60%: the workload is so repetitive you may not need an LLM at all
#   under ~1,000 requests/day: the caching engineering can cost more than it saves

- Each cache hit removes an LLM call, so cost and latency fall in proportion to the **hit rate**.
- At a 45 percent hit rate the bill drops by nearly half and average latency roughly halves - the sweet spot.
- The rules of thumb: target 25 to 45 percent; **above ~60 percent** you may not need an LLM; under ~a thousand requests a day the overhead can exceed the savings.
- The whole rest of the lesson is about *raising* that hit rate safely - catching more repeats without serving a wrong answer.

#### 💡 What just happened

⚡ What just happened? A cache turns repeated LLM calls into near-free lookups; savings scale with the hit rate. And because these apps stream (Lessons 12.1 and 12.2), a hit is even better than the flat latency suggests: it returns the whole answer at once with effectively zero time-to-first-token, versus a streamed miss where the user waits for the first token. Tradeoff / the framing for the lesson: aim for a 25 to 45 percent hit rate, treat a very high one as a signal you may not need an LLM, and skip caching for low-traffic endpoints. Every later step raises the hit rate without breaking correctness.

- Slide the hit rate; watch LLM calls, cost, and latency drop
- The 25-45 percent target band and the '>60 percent you may not need an LLM' line

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 2: Exact-Match Cache: Hash the Normalized Prompt

### Exact-Match Cache: Hash the Normalized Prompt

Normalize the request, hash it to a key, store the response, look it up in O(1). Dead simple and safe - but it only catches identical requests; every paraphrase misses.

The simplest cache, and the one you always build first. **Normalize** the incoming prompt (lowercase it, trim it, collapse internal whitespace, sort any unordered fields) so trivially-different requests map to the same thing, then **hash** the normalized text to a compact key, **store** the response under it, and **look it up** in O(1). This check-the-cache-then-fill-on-a-miss shape - your application owns the lookup and populates the cache itself - is the classic **cache-aside** (lazy-population) pattern, and every cache in this lesson uses it. It is dead simple and completely **safe** - only a genuinely identical request hits, so you can never serve the wrong answer. Its limit is exactly its strength: it keys on the literal text, so it **misses every paraphrase**. “What is the refund policy?” and “How do refunds work?” want the same answer but hash to different keys. That gap is why the next step exists. The block hashes, stores, and looks up, keyless.

> 📱 **Analogy**
>
> It is a **contact list keyed by exact name**. If you saved a friend as “Robert Smith” and later search “Bob Smith,” the phone finds nothing - even though it is the same person - because it matches the exact stored string. Normalizing (ignoring case and spacing) is like ignoring whether you typed “robert smith” or “Robert Smith ”; but “Bob” is a different string entirely, so it still misses. Exact-match caching has exactly this blind spot for paraphrases.

An exact-match cache holds “What is the refund policy?”. A user asks “how do refunds work?”. What happens?

**📝 `02_exact_cache.py`** - *runnable*

In [ ]:
# EXACT-MATCH CACHE - hash the NORMALIZED prompt to a key, store the response, look it up in
# O(1). Dead simple and safe, but it only catches IDENTICAL requests - every paraphrase misses.
# No key.
import hashlib

def normalize(prompt):                         # lowercase, trim, collapse internal whitespace
    return " ".join(prompt.lower().strip().split())

def key(prompt):
    return hashlib.md5(normalize(prompt).encode()).hexdigest()[:10]

cache = {}
def ask(prompt, answer=None):
    k = key(prompt)
    if k in cache:
        return "HIT ", cache[k]
    cache[k] = answer or "(freshly generated)"
    return "MISS", cache[k]

print("Populate the cache, then look it up:")
print(f"  {ask('What is the refund policy?', 'Refunds within 30 days.')[0]}  key={key('What is the refund policy?')}  'What is the refund policy?'")
# An identical request (different case + spacing) normalizes to the SAME key -> HIT.
print(f"  {ask('  what is the REFUND policy?  ')[0]}  key={key('  what is the REFUND policy?  ')}  'what is the REFUND policy?' (same after normalize)")
# A paraphrase asks the same thing but hashes to a DIFFERENT key -> MISS.
print(f"  {ask('How do refunds work?')[0]}  key={key('How do refunds work?')}  'How do refunds work?' (a paraphrase)")
print("\nExact-match hits only identical text. Most repeated questions are worded differently -> Step 3.")
# Output:
# Populate the cache, then look it up:
#   MISS  key=f452fc0bc0  'What is the refund policy?'
#   HIT   key=f452fc0bc0  'what is the REFUND policy?' (same after normalize)
#   MISS  key=7c0c146037  'How do refunds work?' (a paraphrase)
#
# Exact-match hits only identical text. Most repeated questions are worded differently -> Step 3.

- `normalize()` lowercases, trims, and collapses whitespace so trivially-different requests share a key.
- The first ask is a **MISS** (it populates the cache); an identical request with different case and spacing normalizes to the **same key** and **HITS**.
- The paraphrase “How do refunds work?” normalizes to different text, so it produces a **different key** and **MISSES**.
- Exact-match is O(1) and never wrong, but it only catches identical text - and most repeats are worded differently.

#### 💡 What just happened

⚡ What just happened? Exact-match caching hashes the normalized prompt and looks it up in O(1); it is safe but catches only identical requests. Tradeoff: zero false positives, but it misses paraphrases, which are the bulk of real repeats. Normalization widens what counts as identical; semantic caching (next) widens it to meaning.

- A prompt is normalized and hashed to a key; an identical request hits the store
- A paraphrase hashes to a different key and misses

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 3: Semantic Cache: Catch Paraphrases, Mind the False Positives

### Semantic Cache: Catch Paraphrases, Mind the False Positives

Embed the query and return a cached answer if a stored query is within a cosine-similarity threshold. This catches paraphrases - but the false-positive tradeoff is the whole game.

To catch the paraphrases exact-match misses, cache by **meaning**. Embed each query into a vector (as in 11.4), store the response indexed by that vector, and on a new query return the cached answer if a stored vector is within a **cosine-similarity threshold**. Now “how do refunds work?” and “what is the refund policy?” land close together and share an answer. But this power is also the danger, and the **false positive is the whole game**: set the threshold too **loose** and you return a cached answer to a question that is actually *different* (a wrong answer that looks right); too **tight** and you miss real paraphrases and the hit rate collapses. Practitioners keep the false-positive rate low (below roughly 2 percent, or 0.5 percent in regulated settings). And the biggest lever is not the threshold at all - it is the **embedding model**: a weak one maps distinct questions to nearby vectors, so no threshold can separate them. The block shows exactly that with a toy embedder.

> 🏫 **Analogy**
>
> It is a **receptionist who recognizes a rephrased request**. Ask for “the person who handles billing” or “whoever I pay my invoice to,” and a good receptionist sends you to the same desk - they match on *intent*, not exact words. But a careless one might hear “billing” and “building” as close and send you to facilities. That mishearing is the false positive: a confidently-wrong routing. A sharper receptionist (a better embedding model) never confuses the two.

You widen the similarity threshold to catch more paraphrases. What new risk does that create?

**📝 `03_semantic_cache.py`** - *runnable*

In [ ]:
# SEMANTIC CACHE - embed the query and return a cached answer if a stored query is within a
# cosine-similarity THRESHOLD. This catches paraphrases exact-match misses - but the FALSE
# POSITIVE is the whole game: too loose returns a wrong answer to a different question. A toy
# bag-of-words embedder stands in for a real model, so this runs with no key.
import math
STOP = {"what", "whats", "is", "the", "a", "how", "does", "do", "your", "are", "of"}

def embed(text):                               # toy: the set of content words (a real model is far better)
    return {w.strip("?.,") for w in text.lower().replace("'", "").split() if w.strip("?.,") not in STOP}

def cosine(a, b):
    if not a or not b: return 0.0
    return len(a & b) / (math.sqrt(len(a)) * math.sqrt(len(b)))

STORE = [("what is the refund policy", "Refunds are accepted within 30 days.")]
stored_vec = embed(STORE[0][0])

def semantic_lookup(query, threshold):
    v = embed(query)
    sim = cosine(v, stored_vec)
    return sim, ("HIT " if sim >= threshold else "MISS"), STORE[0][1]

queries = ["whats the refund policy?",      # a clean paraphrase (same intent)
           "how does a refund work?",        # a looser paraphrase (same intent)
           "what is the privacy policy?"]    # a DIFFERENT question that shares the word 'policy'
for thr in (0.45, 0.90):
    print(f"threshold {thr}:")
    for q in queries:
        sim, verdict, ans = semantic_lookup(q, thr)
        note = ""
        if verdict == "HIT " and q.startswith("what is the privacy"):
            note = "  <- FALSE HIT: returns the REFUND answer to a PRIVACY question (wrong!)"
        print(f"  sim={sim:.2f} {verdict} {q:32}{note}")
    print()
print("The weak embedder cannot separate 'refund work' from 'privacy policy' (both 0.50): loose")
print("catches the paraphrase but false-hits privacy; tight avoids it but misses the paraphrase.")
print("So the EMBEDDING MODEL matters most - a better one places intents where a threshold can split them.")
# Output:
# threshold 0.45:
#   sim=1.00 HIT  whats the refund policy?
#   sim=0.50 HIT  how does a refund work?
#   sim=0.50 HIT  what is the privacy policy?       <- FALSE HIT: returns the REFUND answer to a PRIVACY question (wrong!)
#
# threshold 0.9:
#   sim=1.00 HIT  whats the refund policy?
#   sim=0.50 MISS how does a refund work?
#   sim=0.50 MISS what is the privacy policy?
#
# The weak embedder cannot separate 'refund work' from 'privacy policy' (both 0.50): loose
# catches the paraphrase but false-hits privacy; tight avoids it but misses the paraphrase.
# So the EMBEDDING MODEL matters most - a better one places intents where a threshold can split them.

- The clean paraphrase “what’s the refund policy?” scores 1.00 and **hits at any threshold** - a real win over exact-match.
- At the loose threshold, “how does a refund work?” (a real paraphrase) hits - but so does “what is the privacy policy?”, a **FALSE HIT** that serves the refund answer to a privacy question.
- At the tight threshold, the false hit is gone - but so is the good paraphrase; both score 0.50 and the weak embedder cannot separate them.
- The lesson: no threshold cleanly splits them here, so the **embedding model matters most** - a better one places intents where a threshold can divide right from wrong.

#### 💡 What just happened

⚡ What just happened? A semantic cache returns a cached answer when a query’s embedding is within a similarity threshold, catching paraphrases. Tradeoff / the whole game: loose = false hits (wrong-but-plausible answers), tight = missed hits. Tune to a low false-positive rate, and remember the embedding model separates right from wrong more than the threshold does. Deep embedding choice is a Module 8 topic.

- Queries as points near a stored vector; a similarity dial sets the threshold
- Too loose false-hits a different question; too tight misses a real paraphrase

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 4: Provider Prompt Caching: Cache the Prefix, 90 Percent Off Input

### Provider Prompt Caching: Cache the Prefix, 90 Percent Off Input

A different cache entirely: the provider caches a long stable prompt prefix so the next call’s cached INPUT tokens cost about a tenth of the normal rate. It stacks with response caching, it does not replace it.

Steps 2 and 3 cache the *response*. This step is a completely different mechanism that the providers give you for free-ish in 2026: **prompt caching**, which caches the *input*. When you send a long, stable **prefix** - a big system prompt, a document, a set of few-shot examples - the provider can cache it so that on the next call, those cached **input tokens are billed at roughly 0.1x** (about a **90 percent discount**) and served faster. All three major providers offer it, with different ergonomics: **Anthropic** is manual (a `cache_control` breakpoint on the prefix, with a small cache-write surcharge and a short TTL, its time-to-live before it expires - covered in Step 5); **OpenAI** is automatic for prefixes over about a thousand tokens; **Gemini** offers an explicit named cache and an implicit automatic one. The rule is simple: put the **stable content first**, because the cache matches on the prefix up to the first thing that changes. This is *orthogonal* to response caching - you use both. The block models the prefix-reuse economics.

> 🍳 **Analogy**
>
> It is a restaurant **prepping the base sauce once for every dish**. A good kitchen does not chop onions and simmer the stock from scratch for every single order - it makes a big batch of the base once in the morning, then finishes each plate with the bit that differs. The base is your long system prompt (cached, reused cheaply); the finishing touch is the user’s specific question (always fresh). Provider prompt caching is that shared mise en place: pay to prep the prefix once, reuse it for pennies.

You send the same 10,000-token system prompt on 5 calls. With provider prompt caching, roughly what happens to the input cost?

**📝 `04_prompt_caching.py`** - *runnable*

In [ ]:
# PROVIDER PROMPT CACHING - a DIFFERENT cache: the provider caches a long, stable prompt PREFIX
# (a big system prompt or document) so the next call's cached INPUT tokens cost ~0.1x (a 90%
# discount). It caches input, not the response - orthogonal to Steps 2-3. Illustrative prices/tokens.

PREFIX_TOK, SUFFIX_TOK, CALLS = 10_000, 200, 5     # a 10k-token system prompt reused across 5 calls
PRICE = 3.0 / 1_000_000                             # $ per input token (illustrative)
WRITE_MULT, READ_MULT = 1.25, 0.1                   # cache write 1.25x, cache read 0.1x (Anthropic-style)

no_cache = CALLS * (PREFIX_TOK + SUFFIX_TOK) * PRICE        # every call pays full price for the whole prefix
# With prompt caching: call 1 WRITES the prefix (1.25x); calls 2..N READ it (0.1x). Suffix always full price.
write_cost = (PREFIX_TOK * WRITE_MULT + SUFFIX_TOK) * PRICE
read_cost  = (CALLS - 1) * (PREFIX_TOK * READ_MULT + SUFFIX_TOK) * PRICE
cached = write_cost + read_cost

print(f"A {PREFIX_TOK:,}-token prefix reused across {CALLS} calls (each + a {SUFFIX_TOK}-token suffix):")
print(f"  no prompt caching: every call pays full input price   -> ${no_cache:.4f}")
print(f"  with prompt caching: call 1 writes (1.25x), calls 2-5 read (0.1x) -> ${cached:.4f}")
print(f"  saved: ${no_cache - cached:.4f}  ({1 - cached/no_cache:.0%} off the input bill)")
print("\nKey: put the STABLE content first (it is cached by prefix); the cache matches up to the first change.")
print("Anthropic: manual cache_control + a write surcharge. OpenAI: automatic for prefixes >= 1,024 tokens.")
# Output:
# A 10,000-token prefix reused across 5 calls (each + a 200-token suffix):
#   no prompt caching: every call pays full input price   -> $0.1530
#   with prompt caching: call 1 writes (1.25x), calls 2-5 read (0.1x) -> $0.0525
#   saved: $0.1005  (66% off the input bill)
#
# Key: put the STABLE content first (it is cached by prefix); the cache matches up to the first change.
# Anthropic: manual cache_control + a write surcharge. OpenAI: automatic for prefixes >= 1,024 tokens.

- Without prompt caching, every call pays **full input price for the whole 10,000-token prefix** - five times over.
- With prompt caching, call 1 **writes** the prefix (a small surcharge, ~1.25x) and calls 2 through 5 **read** it at ~0.1x.
- The result is a large cut to the input bill - here about two-thirds off - for content you were sending anyway.
- This caches the **input prefix**, not the response, so it stacks with the exact and semantic caches; put the stable content first so the prefix matches.

#### 💡 What just happened

⚡ What just happened? Provider prompt caching discounts the cached INPUT tokens of a reused prefix by about 90 percent (Anthropic cache_control, OpenAI automatic, Gemini explicit + implicit). Tradeoff: a small one-time write surcharge and a short TTL, in exchange for a large recurring input discount. It is orthogonal to response caching - put the stable content first and use both.

- A long prefix + a short suffix; the first call writes the prefix cache (1.25x), later calls read it (0.1x)
- A cost bar shrinks versus paying full price every call

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 5: TTL and Eviction: Bound the Cache

### TTL and Eviction: Bound the Cache

A cache is finite, so entries expire on a time-to-live and are evicted by a policy - LRU is the common default. A bigger cache lifts the hit rate but costs memory; a shorter TTL keeps answers fresher.

A real cache cannot grow forever, so it needs two ways to let go of entries. A **TTL** (time-to-live) expires an entry after a set duration, which bounds how *stale* a cached answer can get. And when the cache is **full**, an **eviction policy** decides what to drop to make room: **LRU** (least-recently-used) is the common default - evict whatever has gone longest without being read, on the bet it is least likely to be needed next - with **LFU** (least-frequently-used) and plain TTL expiry as alternatives. The two dials trade off directly: a **bigger cache** holds more entries and lifts the hit rate but costs more memory; a **shorter TTL** keeps answers fresh but lowers the hit rate. The block runs an LRU cache with a TTL, evicting and expiring.

> 🧊 **Analogy**
>
> It is a **fridge with expiry dates and limited shelves**. Two rules keep it usable: things past their expiry date get thrown out no matter what (that is the TTL), and when the shelves are full and you buy something new, you toss whatever has been shoved at the back untouched the longest (that is LRU eviction). A bigger fridge holds more (a higher hit rate) but costs more; shorter expiry dates keep everything fresh but mean you re-buy more often.

An LRU cache is full. You just read entry A, and now a new entry D arrives. Which entry is evicted?

**📝 `05_ttl_eviction.py`** - *runnable*

In [ ]:
# TTL & EVICTION - a cache is finite, so entries EXPIRE on a time-to-live and are EVICTED by a
# policy. LRU (least-recently-used) is the common default. A tick clock stands in for time. No key.
from collections import OrderedDict

class LRUCacheTTL:
    def __init__(self, capacity, ttl):
        self.cap, self.ttl, self.store = capacity, ttl, OrderedDict()   # key -> (value, inserted_tick)
    def get(self, key, now):
        if key not in self.store: return None
        value, ins = self.store[key]
        if now - ins > self.ttl:                                        # expired
            del self.store[key]; return "EXPIRED"
        self.store.move_to_end(key)                                     # mark most-recently-used
        return value
    def put(self, key, value, now):
        if key in self.store: self.store.move_to_end(key)
        self.store[key] = (value, now)
        if len(self.store) > self.cap:                                  # evict least-recently-used
            evicted, _ = self.store.popitem(last=False)
            return evicted
        return None

c = LRUCacheTTL(capacity=3, ttl=10)
print("LRU cache, capacity 3, TTL 10 ticks:")
for k in "ABC": c.put(k, f"ans_{k}", now=0)
print(f"  inserted A,B,C -> cache holds {list(c.store)}")
c.get("A", now=1)                                                       # A is now most-recently-used
print(f"  read A (A is now MRU)          -> order {list(c.store)}")
ev = c.put("D", "ans_D", now=2)
print(f"  insert D (over capacity)       -> evicted {ev} (the least-recently-used), holds {list(c.store)}")
print(f"  read C at tick 15 (> TTL 10)   -> {c.get('C', now=15)} (expired and dropped)")
print("\nTradeoff: a bigger cache lifts the hit rate but costs memory; a shorter TTL keeps answers fresh.")
# Output:
# LRU cache, capacity 3, TTL 10 ticks:
#   inserted A,B,C -> cache holds ['A', 'B', 'C']
#   read A (A is now MRU)          -> order ['B', 'C', 'A']
#   insert D (over capacity)       -> evicted B (the least-recently-used), holds ['C', 'A', 'D']
#   read C at tick 15 (> TTL 10)   -> EXPIRED (expired and dropped)
#
# Tradeoff: a bigger cache lifts the hit rate but costs memory; a shorter TTL keeps answers fresh.

- The cache fills with A, B, C at capacity 3; reading **A moves it to most-recently-used**, so the order becomes B, C, A.
- Inserting D exceeds capacity, so LRU **evicts B** - the least-recently-used - leaving C, A, D.
- Reading C after its **TTL has passed** returns EXPIRED and drops it, regardless of recency.
- Two dials: capacity trades memory for hit rate; TTL trades hit rate for freshness.

#### 💡 What just happened

⚡ What just happened? A cache bounds itself with a TTL (expire by age) and an eviction policy (LRU drops the least-recently-used when full). Tradeoff: a bigger cache and a longer TTL lift the hit rate but cost memory and risk staleness; a smaller cache and a shorter TTL stay fresh but hit less. Tune both to your traffic and your tolerance for stale answers - which is exactly the next step.

- An LRU cache of a few slots; a new entry pushes out the least-recently-used
- A TTL timer expires an entry on lookup

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 6: Invalidation and Staleness: The Hard Problem

### Invalidation and Staleness: The Hard Problem

A stale cached answer still looks valid; it is just wrong. Cache keys carry scope and version, an entry is invalid when any of those change, and some answers must never be cached at all.

Here is the genuinely hard part, the one the old joke is about (“the two hard problems in computer science are cache invalidation and naming things”). With LLMs it is *harder*, because a **stale cached answer still looks like a valid answer - it is just wrong**, and nothing about it signals that it is out of date. The discipline has two halves. First, **version your keys**: a cache key must carry the request’s **scope** (which tenant or user) and its **version** (the model, the system-prompt version, the source document version), so that a model upgrade, a prompt edit, or a data change produces a *new* key and the old entry is never served. Second, know **when NOT to cache at all**: personalized answers, time-sensitive answers, stateful answers, and high-stakes answers should be skipped rather than cached. The block shows a versioned key invalidating correctly, and an un-versioned key serving a dangerous stale hit.

> 🥛 **Analogy**
>
> It is a **carton of milk that looks fine but has gone off**. Nothing about the carton tells you it is bad - it looks and pours exactly like fresh milk, right up until you taste it. A stale cache hit is that carton: a perfectly plausible answer that happens to be wrong because the world moved on. The expiry date printed on the carton is your version key - it lets you throw the milk out *before* you drink it, instead of finding out the hard way.

You upgrade your model, but your cache key is just a hash of the prompt text. What does the cache serve on the next repeat?

**📝 `06_invalidation.py`** - *runnable*

In [ ]:
# INVALIDATION & STALENESS - the hard problem. A stale cached answer still LOOKS valid; it is
# just wrong. The fix: cache keys carry SCOPE (tenant/user) and VERSION (model, prompt, source), and
# an entry is invalid when any of those change. No key.
import hashlib

def versioned_key(prompt, model, prompt_ver, source_ver):
    raw = f"{prompt}|{model}|{prompt_ver}|{source_ver}"
    return hashlib.md5(raw.encode()).hexdigest()[:10]

cache = {}
def ask(prompt, model, prompt_ver, source_ver, answer):
    k = versioned_key(prompt, model, prompt_ver, source_ver)
    if k in cache: return "HIT ", cache[k]
    cache[k] = answer; return "MISS", cache[k]

print("A versioned, scoped cache key invalidates automatically when anything changes:")
r = ask("price?", "claude-sonnet-4-6", "v1", "prices-2026-06", "The course is 14999.")
print(f"  {r[0]} model=claude-sonnet-4-6 prompt=v1 source=prices-2026-06 -> {r[1]}")
r = ask("price?", "claude-sonnet-4-6", "v1", "prices-2026-06", "...")          # same versions
print(f"  {r[0]} identical versions -> cache HIT (correct)")
r = ask("price?", "claude-opus-4-8",   "v1", "prices-2026-06", "The course is 14999 (opus).")
print(f"  {r[0]} model upgraded -> DIFFERENT key, old entry not served (correctly invalidated)")

# The danger: an UN-versioned key that ignores the source version.
print("\nAn un-versioned key serves a STALE hit after the source changes:")
naive = {}
naive[hashlib.md5(b'price?').hexdigest()[:10]] = "The course is 14999."          # cached under the OLD price
# ... the price is updated to 19999, but the naive key is the same, so:
served = naive[hashlib.md5(b'price?').hexdigest()[:10]]
print(f"  price updated to 19999, but the un-versioned key returns: '{served}'  <- STALE: looks valid, is WRONG")
print("Rule: key on model + prompt + source version; and NEVER cache personalized or time-sensitive answers.")
# Output:
# A versioned, scoped cache key invalidates automatically when anything changes:
#   MISS model=claude-sonnet-4-6 prompt=v1 source=prices-2026-06 -> The course is 14999.
#   HIT  identical versions -> cache HIT (correct)
#   MISS model upgraded -> DIFFERENT key, old entry not served (correctly invalidated)
#
# An un-versioned key serves a STALE hit after the source changes:
#   price updated to 19999, but the un-versioned key returns: 'The course is 14999.'  <- STALE: looks valid, is WRONG
# Rule: key on model + prompt + source version; and NEVER cache personalized or time-sensitive answers.

- The **versioned key** mixes the prompt with the model, prompt, and source versions, so an identical request with identical versions HITS correctly.
- Upgrading the model changes the key, so the old entry is **not served** - the cache invalidated itself automatically.
- The **un-versioned key** ignores the source version, so after the price changes it still returns the OLD price - a **stale hit that looks valid but is wrong**.
- The rule: key on model + prompt + source version, and never cache personalized or time-sensitive answers at all.

#### 💡 What just happened

⚡ What just happened? Cache invalidation is the hard problem because a stale hit looks valid but is wrong. Fix it by versioning keys (scope + model + prompt + source) so a change produces a new key, and by refusing to cache personalized, time-sensitive, or high-stakes answers. Tradeoff: more key fields and more invalidation logic, in exchange for never confidently serving an outdated answer.

- A source doc changes; a version-keyed cache invalidates the old entry
- An un-versioned cache serves a stale hit that looks valid but is wrong

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

---

## 🎯 Concept 7: The Multi-Tier Cache Stack

### The Multi-Tier Cache Stack

Compose the layers in cost order: exact-match first (O(1)), then semantic, then the provider prompt cache for the prefix, and only on a full miss does the real LLM call run and populate the caches.

Now assemble everything into one **stack**, checking layers in **cost order** so the cheapest check runs first. An incoming request hits the **exact-match** cache first (an O(1) hash lookup); on a miss it tries the **semantic** cache (an embedding + similarity search, more expensive but catches paraphrases); the **provider prompt cache** is doing its own job underneath for the shared prefix; and only on a **full miss** does the real **LLM call** run - after which you populate *both* the exact and semantic caches so the next repeat is cheap. And because you now have a system, you **measure it**: the hit rate per layer, the total hit rate, and the cost and latency saved. That is how you know the stack is working and where to tune it. The block composes the stack over a request log and reports the metrics.

> ☕ **Analogy**
>
> It is a **coffee shop checking the counter, then the back, then brewing**. When you order, the barista first glances at the ready-made cups on the counter (exact-match - instant); if none match, they check the urn brewed a few minutes ago in the back (semantic - close enough); and only if both come up empty do they grind beans and brew a fresh pot (the LLM call), setting some aside on the counter for the next person. Fastest check first, brew only when you must.

A request misses the exact-match cache. What should the stack check before calling the LLM?

**📝 `07_cache_stack.py`** - *runnable*

In [ ]:
# THE MULTI-TIER CACHE STACK - compose the layers in cost order: exact-match (O(1)) first, then a
# semantic check (catches paraphrases), and only on a full MISS does the real LLM call run and
# populate both caches. Track per-layer hits and the overall hit rate + cost saved. No key.
import math, hashlib
STOP = {"what", "whats", "is", "the", "a", "how", "does", "do", "your", "of"}
def embed(t): return {w.strip("?.,") for w in t.lower().replace("'","").split() if w.strip("?.,") not in STOP}
def cosine(a, b): return 0.0 if not a or not b else len(a & b)/(math.sqrt(len(a))*math.sqrt(len(b)))
def norm(p): return " ".join(p.lower().strip().split())

exact, semantic = {}, []           # semantic: list of (vec, answer)
THRESHOLD, LLM_COST = 0.8, 0.002
stats = {"exact": 0, "semantic": 0, "llm": 0, "cost": 0.0}

def answer(prompt):
    k = hashlib.md5(norm(prompt).encode()).hexdigest()[:8]
    if k in exact:                                  # tier 1: exact-match
        stats["exact"] += 1; return "exact", exact[k]
    v = embed(prompt)
    for vec, ans in semantic:                       # tier 2: semantic
        if cosine(v, vec) >= THRESHOLD:
            stats["semantic"] += 1; exact[k] = ans; return "semantic", ans
    resp = f"LLM answer to '{prompt[:24]}'"          # tier 3: MISS -> call the LLM, populate both
    stats["llm"] += 1; stats["cost"] += LLM_COST
    exact[k] = resp; semantic.append((v, resp)); return "llm (miss)", resp

log = ["What is the refund policy?",     # miss -> LLM
       "What is the refund policy?",     # exact hit
       "what is the refund policy",      # exact hit (normalized)
       "What's the refund policy?",      # semantic hit (paraphrase)
       "How do I reset my password?"]    # miss -> LLM
print("A request log through exact -> semantic -> LLM:")
for p in log:
    tier, _ = answer(p)
    print(f"  {tier:12} <- {p}")
served_from_cache = stats["exact"] + stats["semantic"]
print(f"\nlayer hits: exact={stats['exact']} semantic={stats['semantic']} llm-miss={stats['llm']}")
print(f"hit rate: {served_from_cache}/{len(log)} = {served_from_cache/len(log):.0%}  |  LLM cost ${stats['cost']:.3f} "
      f"(vs ${len(log)*LLM_COST:.3f} uncached, {1 - stats['cost']/(len(log)*LLM_COST):.0%} saved)")
# Output:
# A request log through exact -> semantic -> LLM:
#   llm (miss)   <- What is the refund policy?
#   exact        <- What is the refund policy?
#   semantic     <- what is the refund policy
#   semantic     <- What's the refund policy?
#   llm (miss)   <- How do I reset my password?
#
# layer hits: exact=1 semantic=2 llm-miss=2
# hit rate: 3/5 = 60%  |  LLM cost $0.004 (vs $0.010 uncached, 60% saved)

- Each request threads the layers in order: **exact-match** (O(1)), then **semantic**, then the **LLM** only on a full miss.
- A repeated question is an **exact hit**; a paraphrase (or a punctuation change) that exact-match misses is caught by the **semantic hit**.
- Every miss calls the LLM once and **populates both caches**, so the next repeat is cheap.
- The metrics close the loop: the per-layer hits, the overall hit rate, and the cost saved - here about 60 percent off the uncached bill.

**📝 `07b_gptcache_cache_control.py`** - *illustrative*

In [ ]:
# THE PRODUCTION STACK - a semantic cache + provider prompt caching (illustrative minimal example).
# You do not hand-roll these in production: wrap the client with GPTCache (or Redis LangCache) for
# semantic caching, and add cache_control for provider prompt caching.

# --- Semantic cache with GPTCache (wraps the client in ~2 lines) ---
# pip install gptcache
# from gptcache import cache
# from gptcache.adapter import openai              # a drop-in OpenAI adapter
# cache.init()                                      # embedding + vector store + similarity, handled for you
# cache.set_openai_key()
# Your chat-completions call now checks the semantic cache BEFORE hitting the model.

# --- Redis LangCache: a caching-optimized embedder + a distance threshold ---
#   from langcache import LangCache
#   lc = LangCache(redis_url=..., embedding="redis/langcache-embed-v1", distance_threshold=0.15)
#   hit = lc.search(query)            # returns a cached answer if it is within the threshold

# --- Provider prompt caching: cache a long stable PREFIX (Anthropic) ---
# from anthropic import Anthropic
# client = Anthropic()
# client.messages.create(
#     model="claude-sonnet-4-6",
#     system=[{"type": "text", "text": LONG_SYSTEM_PROMPT,
#              "cache_control": {"type": "ephemeral"}}],      # cache this prefix -> ~90% cheaper on reuse
#     messages=[{"role": "user", "content": user_question}])
# Output: (illustrative minimal example - needs `pip install gptcache anthropic` + provider keys.)
# GPTCache/LangCache do semantic caching (the response); cache_control caches the INPUT prefix (~90% off).

- **GPTCache** wraps the client so a semantic-cache check happens before every model call - Steps 2-3 as a library.
- **Redis LangCache** uses a caching-optimized embedder and a distance threshold - the semantic cache, productized.
- **cache_control** on a system-prompt block is provider prompt caching - Step 4, marking the stable prefix for a ~90 percent input discount.
- In production you compose these; the multi-tier logic you built by hand is what these tools run for you.

#### 💡 What just happened

⚡ What just happened? The stack composes exact -> semantic -> LLM in cost order, populating both caches on a miss, and measures the per-layer and overall hit rate, cost, and latency. Tradeoff / the whole lesson: more layers and metrics to run, but a much higher safe hit rate than any single cache. In production GPTCache / Redis LangCache and provider cache_control run these layers for you.

- A request falls through exact -> semantic -> LLM; each layer lights on a hit
- A running hit-rate and cost-saved meter

> 🎬 *Interactive animation in the HTML lesson: **Interactive animation**. Open the HTML version to interact with it.*

## Putting It Together

> ✅ **Info**
>
> 🧠 The whole picture
> Caching is the highest-leverage cost and latency win in a production AI app, and it is a **layered** discipline. The **economics** set the target: aim for a 25 to 45 percent hit rate, and treat a very high one as a sign you may not need an LLM (Step 1). An **exact-match** cache catches identical requests in O(1) (Step 2); a **semantic** cache catches paraphrases, at the price of tuning a threshold to a low false-positive rate (Step 3); and **provider prompt caching** discounts the reused input prefix by about 90 percent, orthogonally (Step 4). You bound the cache with **TTL and LRU eviction** (Step 5), keep it correct by **versioning keys and refusing to cache what must not be** (Step 6), and compose it all into a **multi-tier stack** checked in cost order, measured end to end (Step 7). The same handful of questions, paid for once instead of a thousand times.

| Cache type | Keys on | Catches | Main risk |
|---|---|---|---|
| Exact-match | hash of the normalized prompt | identical requests (O(1)) | misses every paraphrase |
| Semantic | query embedding + threshold | paraphrases, same intent | false hits if the threshold is loose |
| Provider prompt caching | a stable input prefix | the reused prefix (input tokens) | short TTL; a small write surcharge |
| KV cache(serving layer) | attention keys/values in a decode | token reuse within a generation | infra-level; not app-controllable |

> 📦 **Info**
>
> ➡️ Where this goes nextYou can now cut cost and latency with a layered cache. **Scaling the cache backend** - a Redis cluster or Memorystore that autoscales with traffic - you meet in Lesson 12.7, and **monitoring** the hit rate, false-positive rate, and cost savings comes in Lesson 12.8. The **embedding model** that most determines your semantic-cache quality is covered in Module 8, and **KV caching** at the serving layer comes in Module 9 and Module 14. The primary references are [Anthropic prompt caching](https://docs.claude.com/en/docs/build-with-claude/prompt-caching), [OpenAI prompt caching](https://platform.openai.com/docs/guides/prompt-caching), [Gemini context caching](https://ai.google.dev/gemini-api/docs/caching), [GPTCache](https://github.com/zilliztech/GPTCache), and [Redis LangCache](https://redis.io/tutorials/semantic-caching-with-redis-langcache/).

*Practice exercises are in the companion practice notebook.*

---

## 🎓 Summary

You've completed the practical part of **CachingStrategies**.

### Next steps:
1. Re-run cells with different parameters to build intuition
2. Try the practice exercises (see `practice-lab/practice-lab-12_4.ipynb` if available)
3. Review the full HTML lesson for the complete narrative

*Generated from `lesson-12.4.html` - regenerate this notebook whenever the source HTML is updated.*
